In [1]:
# Put scripts/ on the path so `eval.*` and `fifedata`/`load` import regardless of
# where the kernel started. Paths in the figure cells are relative to this notebook
# (../output/), so the working directory is left alone. Run this cell first.
import os
import pathlib
import sys

_ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / "eval").is_dir())
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))
os.makedirs(_ROOT / "output", exist_ok=True)
print(f"project root: {_ROOT}  (cwd unchanged: {pathlib.Path.cwd()})")


project root: /opt/nfs/home/amaustin/Research/fife-batch-jobs/scripts  (cwd unchanged: /opt/nfs/home/amaustin/Research/fife-batch-jobs/scripts/notebooks)


### Feature engineering

In [2]:
from load import load_jobs

# `lf` and the label expressions are defined once in scripts/load.py, which is
# also what data-analysis.ipynb uses. This notebook can therefore be run on its own.
globals().update(load_jobs())
print(f"lf ready: {len(lf.collect_schema().names())} columns")


Found 168 Parquet partition files.
Total dataset size on disk: 15.60 GB across 168 files.
Columns in NEED not present in raw dataset: ['Group']
Normalised 8 epoch-ms columns (0 -> null): QDate_ms, JobStartDate_ms, JobCurrentStartDate_ms, JobCurrentStartExecutingDate_ms, CompletionDate_ms, EnteredCurrentStatus_ms, LastMatchTime_ms, x509UserProxyExpiration_ms
Resolved: queue=QDate_ms start=JobStartDate_ms completion=CompletionDate terminal-status=EnteredCurrentStatus_ms
Lazy execution plan initialized successfully!
lf ready: 58 columns


In [3]:
import torch 

# --- Configuration & Split Windows ---
TRAIN_YM = [202502, 202503, 202504, 202505, 202506]  # Feb - Jun
TEST_YM  = [202507, 202508]                          # Jul - Aug
MIN_QDATE = "2025-02-01"                             # Drop pre-window stragglers

_GPU = torch.cuda.is_available()
DEVICE = torch.device("cuda" if _GPU else "cpu")
XGB_DEV = "cuda" if _GPU else "cpu"
CB_TASK = "GPU" if _GPU else "CPU"
LGBM_DEV = "cpu"

# Feature Schema
SUB_CAT = ["Group", "Owner", "PomsLauncher", "CampaignName", "CampaignStageName", 
           "CampaignId", "CampaignStageId", "CampaignType", "TestLaunch", "JobsubGroup", 
           "Image", "BlacklistSites"]
SUB_NUM = ["RequestCpus", "RequestDisk", "RequestMemory", "RequestSlots", 
           "ExecutableSize", "TransferInputMB", "ExpectedLifetime", "TotalSubmitProcs"]
MATCH_CAT = ["MatchSite", "MatchEntry", "MatchQueue", "MatchResource", "MatchCpus"]
MATCH_NUM = ["CpusProvisioned", "DiskProvisioned", "MemoryProvisioned"]
CAT_ALL = SUB_CAT + MATCH_CAT
NUM_ALL = SUB_NUM + MATCH_NUM

# Standard column mapping
CAND = {
    "Group": ["Group", "AccountingGroup"], "Owner": ["Owner"],
    "PomsLauncher": ["POMS_LAUNCHER", "POMS4_LAUNCHER"], "CampaignName": ["POMS4_CAMPAIGN_NAME"],
    "CampaignStageName": ["POMS4_CAMPAIGN_STAGE_NAME"], "CampaignId": ["POMS4_CAMPAIGN_ID"],
    "CampaignStageId": ["POMS4_CAMPAIGN_STAGE_ID"],
    "CampaignType": ["DerivedCampaignType"],
    "TestLaunch": ["POMS4_TEST_LAUNCH"], "JobsubGroup": ["Jobsub_Group"],
    "Image": ["SingularityImage"], "BlacklistSites": ["Blacklist_Sites"],
    "MatchSite": ["MATCH_EXP_JOB_GLIDEIN_Site", "MATCH_GLIDEIN_Site"],
    "MatchEntry": ["MATCH_GLIDEIN_Entry_Name"], "MatchQueue": ["MATCH_GLIDEIN_SiteWMS_Queue"],
    "MatchResource": ["MachineAttrGLIDEIN_ResourceName0"], "MatchCpus": ["MachineAttrCpus0"],
    "Node": ["LastRemoteHost", "RemoteHost"],
    "RequestCpus": ["RequestCpus"], "RequestDisk": ["RequestDisk"], "RequestMemory": ["RequestMemory"],
    "RequestSlots": ["RequestSlots"], "ExecutableSize": ["ExecutableSize"],
    "TransferInputMB": ["TransferInputSizeMB"], "ExpectedLifetime": ["JOB_EXPECTED_MAX_LIFETIME"],
    "TotalSubmitProcs": ["TotalSubmitProcs"], "CpusProvisioned": ["CpusProvisioned"],
    "DiskProvisioned": ["DiskProvisioned"], "MemoryProvisioned": ["MemoryProvisioned"],
    "QDate": ["QDate_ms", "QDate"],
    "JobStart": ["JobStartDate_ms", "JobStartDate", "JobCurrentStartDate_ms", "JobCurrentStartDate"],
    "CompletionDate": ["CompletionDate_ms", "CompletionDate"],
}

### Deriving CampaignType from stage name

`POMS4_CAMPAIGN_TYPE` is null for every row in the raw data (100% -- verified
against all 32 partition files), including the ~25% of jobs that have real
`POMS4_CAMPAIGN_ID`/`NAME`/`STAGE_NAME` values. It was never populated
upstream; this isn't something earlier cells in this notebook broke.

Per the "Campaign Type" table in `FIFE-Docs.md`, the type can be recovered
from keywords in `POMS4_CAMPAIGN_STAGE_NAME` instead. Jobs with no POMS4_*
fields at all aren't part of any campaign -- these get `"User"`.

In [4]:
import polars as pl 

# Campaign-type keyword table, mirroring the "Campaign Type" table in
# FIFE-Docs.md. Edit here (and keep the doc in sync) to reclassify a
# stage-name substring.
CAMPAIGN_TYPE_KEYWORDS = {
    "Generation": ["gen", "dio", "endpoint", "corsika", "sim", "wiremod", "ly", "offset",
                   "spill", "g4", "beamgun", "decay", "surface", "cryo"],
    "Reconstruction": ["stage0", "stage1", "reco", "reco1", "reco2", "digi", "track", "decode",
                       "recluster", "fullproduction", "ndlar", "compress", "convert", "fmatch"],
    "Merging": ["merge", "skim", "hadd", "filter", "scrub", "watchdog", "sleep", "test",
                "fclless", "concat", "mix"],
    "Analysis": ["ana", "caf", "ntuple", "larcv"],
}


def _classify_campaign_type(stage_name):
    """Composite stage names (e.g. 'gen_g4_detsim_reco1_reco2_caf') match
    keywords from several categories at once. We take the LAST (rightmost)
    match, on the assumption that the terminal step is what the stage
    actually delivers -- that example ends in 'caf' -> Analysis, even though
    it runs gen/reco steps first. This tie-break decides ~2.98M jobs across
    45 composite stage names, so it's worth a sanity check against how
    these have been labeled by hand in the past.
    """
    low = stage_name.lower()
    best_cat, best_pos = None, -1
    for cat, kws in CAMPAIGN_TYPE_KEYWORDS.items():
        for kw in kws:
            pos = low.find(kw)
            if pos > best_pos:
                best_pos, best_cat = pos, cat
    return best_cat or "Unmapped"


# Build the lookup once over the distinct stage names (~164), then map the
# whole column against it -- far cheaper than a per-row Python call.
_distinct_stages = (
    lf.select(pl.col("POMS4_CAMPAIGN_STAGE_NAME").unique())
    .collect()["POMS4_CAMPAIGN_STAGE_NAME"].drop_nulls().to_list()
)
_stage_to_type = {s: _classify_campaign_type(s) for s in _distinct_stages}

# No POMS4 fields at all -> not part of a campaign -> a user job.
lf = lf.with_columns(
    pl.when(pl.col("POMS4_CAMPAIGN_STAGE_NAME").is_null())
      .then(pl.lit("User"))
      .otherwise(
          pl.col("POMS4_CAMPAIGN_STAGE_NAME").replace_strict(_stage_to_type, default="Unmapped")
      )
      .alias("DerivedCampaignType")
)

# Register the derived column in `have`, the same way cell 2 does for
# "Group". `have` is what CAND/cat_expr() gate on -- a column present in
# `lf` but missing from `have` is silently replaced by a constant 0, which
# is exactly the all-null CampaignType failure this cell exists to fix.
if "DerivedCampaignType" not in have:
    have.append("DerivedCampaignType")

_diag = lf.group_by("DerivedCampaignType").len().sort("len", descending=True).collect()
_total = _diag["len"].sum()
print("CampaignType distribution:")
for row in _diag.iter_rows(named=True):
    print(f"  {row['DerivedCampaignType']:15s} {row['len']:>12,}  ({100 * row['len'] / _total:5.2f}%)")

# Recover the integer code each label ends up with. cat_expr() hashes the
# string, then cell 17 dense-ranks via np.unique -- which sorts -- so a
# label's final code is the rank of its hash among the hashes of the labels
# actually present. Built from _diag (observed values) rather than a fixed
# list, so it stays correct if a filter ever drops a type entirely.
_present = _diag["DerivedCampaignType"].to_list()
_hashes = (
    pl.DataFrame({"v": _present})
    .select(
        (pl.col("v").cast(pl.Utf8).fill_null("__NA__").hash(seed=0) % 2147483629)
        .cast(pl.Int32).alias("h")
    )["h"].to_list()
)
CAMPAIGN_TYPE_CODES = {
    rank: lab
    for rank, (lab, _) in enumerate(sorted(zip(_present, _hashes), key=lambda kv: kv[1]))
}
print("\nCampaignType code -> label (saved to schema_meta.json):")
for _code, _lab in CAMPAIGN_TYPE_CODES.items():
    print(f"  {_code} -> {_lab}")

_unmapped = sorted(s for s, t in _stage_to_type.items() if t == "Unmapped")
assert not _unmapped, (
    f"{len(_unmapped)} stage name(s) matched no keyword: {_unmapped}. "
    f"Extend CAMPAIGN_TYPE_KEYWORDS above (and FIFE-Docs.md) to cover them."
)
print(f"\nAll {len(_distinct_stages)} distinct stage names mapped; "
      f"{_diag.height} unique CampaignType values.")


CampaignType distribution:
  User              48,063,579  (75.00%)
  Reconstruction     9,253,386  (14.44%)
  Analysis           5,237,589  ( 8.17%)
  Generation         1,462,791  ( 2.28%)
  Merging               71,013  ( 0.11%)

CampaignType code -> label (saved to schema_meta.json):
  0 -> Reconstruction
  1 -> Analysis
  2 -> User
  3 -> Merging
  4 -> Generation

All 164 distinct stage names mapped; 5 unique CampaignType values.


In [5]:
import os
import gc
import sys
import time
import datetime as _dt
import numpy as np
import polars as pl
import torch

def cat_expr(std):
    raw = R.get(std)
    if raw is None or raw not in have:
        return pl.lit(0).cast(pl.Int32).alias("c_" + std)
    return ((pl.col(raw).cast(pl.Utf8).fill_null("__NA__").hash(seed=0) % 2147483629)
            .cast(pl.Int32).alias("c_" + std))

def num_expr(std):
    raw = R.get(std)
    if raw is None or raw not in have:
        return pl.lit(0.0).cast(pl.Float32).alias("n_" + std)
    return pl.col(raw).cast(pl.Float32, strict=False).fill_null(0.0).alias("n_" + std)

R = {std: next((c for c in cands if c in have), None) for std, cands in CAND.items()}

sel_exprs = (
    [cat_expr(c) for c in CAT_ALL] + [cat_expr("Node")]
    + [num_expr(c) for c in NUM_ALL]
    + [to_sec(R["QDate"]).alias("t_q") if R["QDate"] else pl.lit(None).alias("t_q"),
       to_sec(R["JobStart"]).alias("t_s") if R["JobStart"] else pl.lit(None).alias("t_s"),
       to_sec(R["CompletionDate"]).alias("t_c") if R["CompletionDate"] else pl.lit(None).alias("t_c"),
       # Accumulated wall clock. Jobs that ran but were REMOVED never get a
       # CompletionDate, so start + walltime is the only way to date their outcome
       # for the temporal split. Not a training feature -- it is outcome-derived.
       (pl.col("RemoteWallClockTime").cast(pl.Float64, strict=False).alias("t_w")
        if "RemoteWallClockTime" in have else pl.lit(None, dtype=pl.Float64).alias("t_w"))]
    + [pl.col("Failed").cast(pl.Int8), pl.col("hw_fault").cast(pl.Int8),
       pl.col("fault_type").cast(pl.Int8), pl.col("wait_s").cast(pl.Float64),
       # Ran: this job actually started, so it is a valid E1/E3 target. Never-ran
       # jobs stay in the frame as queue context and are masked out downstream.
       pl.col("Ran").cast(pl.Int8),
       # When the job left the queue -- start time if it ran, terminal-status time
       # if it was removed first. Drives the idle-queue-depth feature.
       pl.col("t_queue_exit").cast(pl.Float64)]
)

t0 = time.time()
# `lf` is already restricted to the window by load.load_jobs (QMIN_S), so there is no
# second filter here and no second threshold to drift out of sync.
assert QMIN_S == _dt.datetime.fromisoformat(MIN_QDATE).replace(
    tzinfo=_dt.timezone.utc).timestamp(), (
    f"MIN_QDATE ({MIN_QDATE}) disagrees with load.py's window start; change it in "
    f"load.py (WINDOW_START), which is where the filter is applied")
df = lf.select(sel_exprs).collect()
ntot = df.height
print(f"Loaded {ntot:,} rows into RAM in {time.time() - t0:.1f}s")

# Extract categorical codes
codes = np.column_stack([df["c_" + c].to_numpy() for c in CAT_ALL])
node_k = df["c_Node"].to_numpy()
cards = []
for j in range(codes.shape[1]):
    u, inv = np.unique(codes[:, j], return_inverse=True)
    codes[:, j] = inv.astype(np.int32)
    cards.append(len(u))

# Extract numerical features
X_num = np.column_stack([df["n_" + c].to_numpy() for c in NUM_ALL]).astype(np.float32)
for j in range(X_num.shape[1]):
    s = float(X_num[:, j].std()) or 1.0
    X_num[:, j] = (X_num[:, j] - X_num[:, j].mean()) / s

# Timestamps & Labels
qs = df["t_q"].to_numpy()
jst = df["t_s"].to_numpy()
js = jst  # Alias for start time
comp = df["t_c"].to_numpy()
wall = df["t_w"].to_numpy()
ran = df["Ran"].to_numpy().astype(bool)
qexit = df["t_queue_exit"].to_numpy()
wait_sv = df["wait_s"].to_numpy()
ftype = df["fault_type"].to_numpy()
failed = df["Failed"].to_numpy().astype(np.int8)
hw = df["hw_fault"].to_numpy().astype(np.int8)

# Month split masking
dtq = pl.from_epoch(pl.Series(np.nan_to_num(qs).astype(np.int64)), time_unit="s")
ym = (dtq.dt.year().cast(pl.Int32) * 100 + dtq.dt.month().cast(pl.Int32)).to_numpy()
tr_mask = np.isin(ym, TRAIN_YM)
te_mask = np.isin(ym, TEST_YM)

# The polars frame is not used past this point -- every column needed downstream is
# already a numpy array. Holding it alongside Xmatch/Xsub is several GB of nothing.
del df
gc.collect()

print(f"{ntot:,} total rows | Train ({TRAIN_YM}): {tr_mask.sum():,} | Test ({TEST_YM}): {te_mask.sum():,}")

Loaded 64,088,358 rows into RAM in 105.9s
64,088,358 total rows | Train ([202502, 202503, 202504, 202505, 202506]): 48,551,819 | Test ([202507, 202508]): 15,536,539


In [6]:
# Trailing windows, in seconds.
#
# Trailing windows are chosen per KEY from that key's own inter-failure timescale
# (data-analysis.ipynb `fitsite1`, and the per-key fits in MODELING.md 13.4). A single
# global window list is wrong here, because the three keys operate decades apart:
#
#   key        fitted lambda2/lambda   median gap   chi2/ndof
#   site          7.75 s / 37.96 s          1 s       3.88
#   campaign      8.04 s / 45.79 s          -         4.53
#   node          two-exponential FAILS   415 s    1449      <- different process
#
# site and campaign cluster on tens of seconds, so a 60 s window captures the burst
# (79% of the slow component) at the best signal-to-background of any window covering
# both timescales; 300 s and 900 s hold the same burst with 4.7x and 14x more
# background, so they are dropped. 3600 s stands in for the fit's constant term.
#
# Node failures are ~10,000x more spread out (mean gap 76,158 s vs 7.2 s at site
# level) and the two-exponential form does not fit them at all. A 60 s window contains
# a prior failure on the same node only 15% of the time -- it would be mostly empty.
# Node windows are therefore set from the node gap distribution directly: 3600 s
# covers 64.5% of node gaps and 86400 s covers 83.2%.
TRAIL_WINDOWS_BY_KEY = {
    "site_fail": [60, 3600],
    "site_hw":   [60, 3600],
    "camp_fail": [60, 3600],
    "node_fail": [3600, 86400],
    "node_hw":   [3600, 86400],
}

# entry_fail is dropped: GlideinWMS entries nest inside sites, so it measures almost
# the same thing as site_fail -- r = 0.995 (15m) and 0.996 (60m) on a 2M-row sample.
# The *_hw rates are kept despite near-zero univariate correlation with Failed: they
# carry the largest |r| with the HARDWARE target of any trailing feature (0.040 for
# trail60m_site_hw), which is small only because hardware faults are 0.8% prevalent,
# and they are the sole hardware-specific trailing signal E3 has.
TRAIL_NAMES = list(TRAIL_WINDOWS_BY_KEY)

# Flat (window, statistic) pairs, in column order.
TRAIL_PAIRS = [(w, nm) for nm in TRAIL_NAMES for w in TRAIL_WINDOWS_BY_KEY[nm]]

n_cat, n_num = len(CAT_ALL), len(NUM_ALL)
n_base = n_cat + n_num
n_sc, n_sn = len(SUB_CAT), len(SUB_NUM)

XMATCH_COLS = (CAT_ALL + [c + " (std)" for c in NUM_ALL]
               + ["sin_hour@match", "cos_hour@match", "sin_wday@match", "cos_wday@match"]
               + [f"trail{w // 60}m_{nm}" for w, nm in TRAIL_PAIRS]
               + ["log_site_running@match", "log_total_running@match"])

XSUB_COLS = (SUB_CAT + [c + " (std)" for c in SUB_NUM]
             + ["sin_hour@submit", "cos_hour@submit", "sin_wday@submit", "cos_wday@submit"]
             + ["log_idle_queue_depth", "log_camp_trail_wait", "log_total_running@submit"])

NXM, NXS = len(XMATCH_COLS), len(XSUB_COLS)

# Vectorized helper for trailing failure/hardware rates
_TRAIL_CACHE = {}

def trailing_rate(key, tref, tcomp, outcome, is_comp, window_s, cache_key=None,
                  max_window=None):
    """Leak-free trailing rate of `outcome` over `window_s` seconds, per key.

    Sorting the completed rows dominates the cost and is identical for every
    statistic computed on the same key, so it is cached under `cache_key`. The
    encoding uses `max_window` for its span rather than this call's window, which
    makes the sorted array window-independent and lets the window bounds be cached
    too. Verified bit-identical to the uncached form across all windows and both
    outcome vectors.
    """
    n = len(key); w = int(window_s)
    mw = int(max_window if max_window is not None else w)
    _ck = ("sort", cache_key if cache_key is not None else id(key), mw)
    ent = _TRAIL_CACHE.get(_ck)
    if ent is None:
        ci = np.where(is_comp)[0]
        ck = np.asarray(key)[ci].astype(np.int64)
        ct = np.rint(np.asarray(tcomp)[ci]).astype(np.int64)
        if len(ct) == 0:
            _TRAIL_CACHE[_ck] = ()
            return np.full(n, np.nan, np.float32), np.zeros(n, np.float32)
        o = np.lexsort((ct, ck))
        ci, ck, ct = ci[o], ck[o], ct[o]
        t0 = int(ct.min()); span = int(ct.max()) - t0 + mw + 3
        ent = (ci, ct, t0, span, ck * span + (ct - t0))
        _TRAIL_CACHE[_ck] = ent
        del ck
    if ent == ():
        return np.full(n, np.nan, np.float32), np.zeros(n, np.float32)
    ci, ct_s, t0, span, comp_key = ent

    tref = np.asarray(tref); tcomp = np.asarray(tcomp); outcome = np.asarray(outcome)
    _bk = ("bounds", cache_key if cache_key is not None else id(key), mw, w)
    bnd = _TRAIL_CACHE.get(_bk)
    if bnd is None:
        kq = np.asarray(key).astype(np.int64)
        tq = np.where(np.isnan(tref), t0 - w - 10, np.rint(tref)).astype(np.int64)
        hi = np.searchsorted(comp_key, kq * span + np.clip(tq - t0, -1, span - 2), "right")
        lo = np.searchsorted(comp_key, kq * span + np.clip(tq - w - t0, -1, span - 2), "right")
        bnd = (hi, lo)
        _TRAIL_CACHE[_bk] = bnd
        del kq, tq
    hi, lo = bnd

    csum = np.concatenate([[0.0], np.cumsum(np.asarray(outcome)[ci].astype(np.float64))])
    cnt = (hi - lo).astype(np.float64); ssum = csum[hi] - csum[lo]
    self_in = np.asarray(is_comp) & ~np.isnan(tref) & (tcomp <= tref) & (tcomp > tref - window_s)
    ssum[self_in] -= outcome[self_in]; cnt[self_in] -= 1
    with np.errstate(invalid="ignore", divide="ignore"):
        rate = np.where(cnt > 0, ssum / np.maximum(cnt, 1), np.nan)
    return rate.astype(np.float32), cnt.astype(np.float32)

# Cyclical clock features helper
def cyc_of(t):
    d = pl.from_epoch(pl.Series(np.asarray(t)).cast(pl.Int64, strict=False), time_unit="s")
    h = d.dt.hour().to_numpy().astype(np.float64)
    dw = d.dt.weekday().to_numpy().astype(np.float64)
    return np.nan_to_num(np.column_stack([np.sin(2 * np.pi * h / 24), np.cos(2 * np.pi * h / 24),
                                          np.sin(2 * np.pi * dw / 7), np.cos(2 * np.pi * dw / 7)])).astype(np.float32)

print("Calculating trailing state features...")
comp_ok = ~np.isnan(np.asarray(comp)); H1 = 3600.0
fl64, hw64 = np.asarray(failed).astype(np.float64), hw.astype(np.float64)
jsf = np.where(np.isnan(np.asarray(js)), -1e18, np.asarray(js))
tref_m = np.where(np.isnan(np.asarray(js)), np.asarray(qs), np.asarray(js))

site_k = codes[:, CAT_ALL.index("MatchSite")]
camp_k = codes[:, CAT_ALL.index("CampaignId")]

# Keyed by statistic name, and iterated over TRAIL_PAIRS, so the rate order is the
# column-name order by construction. The previous cross product over all windows x
# all specs cannot express per-key windows -- it would compute 18 rates for 10 named
# columns and silently misalign them.
TRAIL_SPEC = {
    "site_fail": (site_k, jsf, fl64),
    "node_fail": (node_k, jsf, fl64),
    "site_hw":   (site_k, jsf, hw64),
    "node_hw":   (node_k, jsf, hw64),
    "camp_fail": (camp_k, qs,  fl64),
}

_MAXW = max(w for w, _ in TRAIL_PAIRS)
_KEYNAME = {"site_fail": "site", "site_hw": "site", "node_fail": "node",
            "node_hw": "node", "camp_fail": "camp"}

tr_all = []
for w, nm in TRAIL_PAIRS:
    kv, tv, ov = TRAIL_SPEC[nm]
    r, _ = trailing_rate(kv, tv, comp, ov, comp_ok, float(w),
                         cache_key=_KEYNAME[nm], max_window=_MAXW)
    tr_all.append(np.nan_to_num(r))

_named = [f"trail{w // 60}m_{nm}" for w, nm in TRAIL_PAIRS]
assert len(tr_all) == len(_named), (
    f"{len(tr_all)} trailing rates computed but {len(_named)} column names")

_TRAIL_CACHE.clear()
gc.collect()

print("Calculating concurrency features...")
fin2 = ~np.isnan(np.asarray(jst)) & ~np.isnan(np.asarray(comp))
s2 = np.sort(np.asarray(jst)[fin2]); c2 = np.sort(np.asarray(comp)[fin2])

def running_total(t):
    r = (np.searchsorted(s2, t, "right") - np.searchsorted(c2, t, "right")).astype(np.float64)
    self_run = fin2 & (np.asarray(jst) <= t) & (np.asarray(comp) > t)
    r[self_run] -= 1
    return np.clip(r, 0, None)

def running_by_key(key, t):
    ki = np.asarray(key).astype(np.int64)
    ks = ki[fin2]
    st = np.rint(np.asarray(jst)[fin2]).astype(np.int64)
    ct = np.rint(np.asarray(comp)[fin2]).astype(np.int64)
    t0 = int(min(st.min(), ct.min())); span = int(max(st.max(), ct.max())) - t0 + 3
    a_s = np.sort(ks * span + (st - t0)); a_c = np.sort(ks * span + (ct - t0))
    q = ki * span + np.clip(np.rint(t).astype(np.int64) - t0, -1, span - 2)
    r = (np.searchsorted(a_s, q, "right") - np.searchsorted(a_c, q, "right")).astype(np.float64)
    self_run = fin2 & (np.asarray(jst) <= t) & (np.asarray(comp) > t)
    r[self_run] -= 1
    return np.clip(r, 0, None)

run_site = running_by_key(site_k, tref_m)
run_tot_m = running_total(tref_m)
cycS = cyc_of(tref_m)

# Assemble Xmatch ON DISK.
#
# Xmatch (~11 GB) and Xsub (~7 GB) previously lived in RAM alongside the source
# arrays, peaking around 35 GB on a shared 125 GB box -- which is what killed the
# kernel when another user's job took memory at the wrong moment. Writing through an
# open_memmap keeps the output out of RAM and persists it in the same pass, so no
# separate save step is needed (there was not one: the matrices were never written).
import os

from load import WINDOW_START  # noqa: F401  (documents the window this matrix covers)

FEAT_DIR = os.environ.get("FIFE_DATA_ROOT",
                          "/mnt/scratch/fast0/amaustin/datasets/fife")
os.makedirs(FEAT_DIR, exist_ok=True)

print(f"Assembling Xmatch on disk -> {FEAT_DIR}/Xmatch.npy "
      f"({int(ntot) * NXM * 4 / 1e9:.1f} GB)...")
Xmatch = np.lib.format.open_memmap(
    os.path.join(FEAT_DIR, "Xmatch.npy"), mode="w+",
    dtype=np.float32, shape=(int(ntot), NXM))
Xmatch[:, :n_cat] = codes
Xmatch[:, n_cat:n_base] = X_num
Xmatch[:, n_base:n_base + 4] = cycS
for j in range(len(tr_all)):
    Xmatch[:, n_base + 4 + j] = tr_all[j]
Xmatch[:, NXM - 2] = np.log1p(run_site)
Xmatch[:, NXM - 1] = np.log1p(run_tot_m)
del cycS, tr_all, run_site, run_tot_m

# Queue depth & submission features
print("Calculating submission & queue features...")
# Idle queue depth: jobs queued before t, minus jobs that had LEFT the queue by t.
# A job leaves either by starting or by being removed while still idle, and the
# September 2025 extraction dates both (see t_queue_exit). Counting only jobs that
# eventually started -- which is what this did while no removal timestamp existed --
# undercounts contention by omitting the ~11% that were removed from the queue.
_qs = np.asarray(qs, dtype=np.float64); _qx = np.asarray(qexit, dtype=np.float64)
fin = np.isfinite(_qs) & np.isfinite(_qx)
q_fin = np.sort(_qs[fin]); s_fin = np.sort(_qx[fin])
idle = np.clip(np.searchsorted(q_fin, _qs, "right")
               - np.searchsorted(s_fin, _qs, "right"), 0, None).astype(np.float64)
wok = comp_ok & ~np.isnan(np.asarray(wait_sv))
campw, _ = trailing_rate(camp_k, qs, comp, np.nan_to_num(np.asarray(wait_sv)), wok, H1)
run_tot_q = running_total(np.asarray(qs))
cycQ = cyc_of(qs)

# Assemble Xsub on disk, same reasoning.
print(f"Assembling Xsub on disk -> {FEAT_DIR}/Xsub.npy "
      f"({int(ntot) * NXS * 4 / 1e9:.1f} GB)...")
Xsub = np.lib.format.open_memmap(
    os.path.join(FEAT_DIR, "Xsub.npy"), mode="w+",
    dtype=np.float32, shape=(int(ntot), NXS))
Xsub[:, :n_sc] = codes[:, :n_sc]
Xsub[:, n_sc:n_sc + n_sn] = X_num[:, :n_sn]
Xsub[:, n_sc + n_sn:n_sc + n_sn + 4] = cycQ
Xsub[:, -3] = np.log1p(idle)
Xsub[:, -2] = np.log1p(np.nan_to_num(campw))
Xsub[:, -1] = np.log1p(run_tot_q)
del cycQ, idle, campw, run_tot_q, q_fin, s_fin, jsf, fl64, hw64
gc.collect()

# Zero-copy views
X = Xmatch[:, :n_base]
trail = Xmatch[:, n_base + 4:NXM - 2]
cyc_q = Xsub[:, n_sc + n_sn:n_sc + n_sn + 4]

# Flush both to disk before anything else runs. Without this the pages are dirty in
# the page cache and a later crash loses them silently.
Xmatch.flush(); Xsub.flush()

print(f"Xmatch shape: {Xmatch.shape}  (base {n_base} + cyc 4 + trailing {len(TRAIL_PAIRS)} + concurrency 2)")
print(f"Xsub shape:   {Xsub.shape}  (sub cat {n_sc} + sub num {n_sn} + cyc 4 + queue state 3)")
print(f"written to {FEAT_DIR}")
for _f in ("Xmatch.npy", "Xsub.npy"):
    _p = os.path.join(FEAT_DIR, _f)
    print(f"  {_f:12s} {os.path.getsize(_p) / 1e9:6.2f} GB")

Calculating trailing state features...
Calculating concurrency features...
Assembling Xmatch on disk -> /mnt/scratch/fast0/amaustin/datasets/fife/Xmatch.npy (11.3 GB)...
Calculating submission & queue features...
Assembling Xsub on disk -> /mnt/scratch/fast0/amaustin/datasets/fife/Xsub.npy (6.9 GB)...
Xmatch shape: (64088358, 44)  (base 28 + cyc 4 + trailing 10 + concurrency 2)
Xsub shape:   (64088358, 27)  (sub cat 12 + sub num 8 + cyc 4 + queue state 3)
written to /mnt/scratch/fast0/amaustin/datasets/fife
  Xmatch.npy    11.28 GB
  Xsub.npy       6.92 GB


### Saving full data

In [ ]:
ANON_START, ANON_END = "2025-02-01", "2025-09-01"      # [start, end)

# Lives on storage0, not scratch: scratch needs its remaining headroom for the
# regenerated Xmatch/Xsub, which grow now that never-ran rows are retained.
ANON_DIR = "/media/storage0/allison/FIFE-Batch-Queues-anon"
ANON_OUT = f"{ANON_DIR}/fife_anon_{ANON_START}_{ANON_END}.parquet"

# Pseudonyms, not raw hashes. Every identifier is hashed one-way, then DENSE-RANKED
# by ascending hash -- which is exactly what the feature cell does with
# np.unique(..., return_inverse=True) -- and rendered as a readable label.
#
# Labels are 0-based, so Xmatch/Xsub code k IS <prefix>k here -- Owner code 0 is
# "user0". No offset to remember when relating a feature-importance result back to a
# row in this file.
#
# Only low-cardinality columns get string labels. LastRemoteHost (1.2M distinct) and
# ClusterId (20M) stay dense Int32 -- "host1216540" for 64M rows would bloat the file
# for no analytical gain.
ANON_LABEL = {                      # column -> pseudonym prefix
    "Owner": "user", "AccountingGroup": "acct",
    "POMS4_CAMPAIGN_ID": "campaign", "POMS4_CAMPAIGN_NAME": "campname",
    "POMS4_CAMPAIGN_STAGE_NAME": "stage", "POMS4_CAMPAIGN_STAGE_ID": "stageid",
    "Jobsub_Group": "jgroup", "SingularityImage": "image", "Blacklist_Sites": "blacklist",
}
ANON_DENSE = ["LastRemoteHost", "ClusterId"]     # too many distinct values to label
# Keep readable: public grid endpoints and derived categories that the figures label
# by name. These identify institutions, not people.
# Group is AccountingGroup truncated at the first dot, i.e. the experiment or
# collaboration: group_nova, group_dune, group_mu2e. Those are public HEP
# collaborations, not people, so they stay readable -- the figures label by them.
# AccountingGroup itself keeps the username suffix (group_nova.cullenms) and stays
# hashed.
ANON_KEEP = ["MATCH_EXP_JOB_GLIDEIN_Site", "MATCH_EXP_JOB_Site", "MATCH_GLIDEIN_Entry_Name",
             "MATCH_GLIDEIN_SiteWMS_Queue", "MachineAttrGLIDEIN_ResourceName0",
             "DerivedCampaignType", "Group"]
# Free text that embeds a username. The label logic matches on substrings of these
# ("condor_rm", "by user", "held 14 days", ...), so only the name is redacted -- the
# pattern the rules key on survives.
ANON_SCRUB = ["RemoveReason", "LastHoldReason"]

_cols = set(lf.collect_schema().names())

# Pilot flag. A glidein pilot is exactly Cmd == "./glidein_startup.sh" -- this is the
# definition, not a proxy. group_opportunistic was used for this before and is close
# but wrong in both directions: of 71.2M raw rows it misses 30,324 real pilots and
# wrongly claims 10,424 jobs that are not pilots. Cmd itself is a path, not an
# identifier, but it is not needed downstream once the flag exists, so only the
# boolean is carried into the anonymised extract.
# Local frame: this cell must not mutate the shared `lf`, which the training-matrix
# cells above build from.
if "Cmd" in _cols:
    _lf_src = lf.with_columns(
        (pl.col("Cmd") == "./glidein_startup.sh").fill_null(False).alias("IsPilot")
    ).drop("Cmd")
    _cols = set(_lf_src.collect_schema().names())
else:
    _lf_src = lf
    print("  WARNING: Cmd absent from lf -- IsPilot not written. Re-run the raw scan "
          "cell (de36b727) so Cmd is carried through.")

def _anon_hash(c):
    """cat_expr()'s hash -- same seed, so the ranking below matches Xmatch/Xsub."""
    return (pl.col(c).cast(pl.Utf8).fill_null("__NA__").hash(seed=0) % 2147483629).cast(pl.Int64)

_label_cols = [c for c in ANON_LABEL if c in _cols]
_dense_cols = [c for c in ANON_DENSE if c in _cols]
_lf_anon = _lf_src.with_columns([_anon_hash(c).alias(c) for c in _label_cols + _dense_cols])

# One pass to collect every column's sorted distinct hashes, then map to pseudonyms.
if _label_cols:
    _uniq = _lf_anon.select(
        [pl.col(c).unique().sort().implode().alias(c) for c in _label_cols]
    ).collect()
    _maps = {}
    for c in _label_cols:
        _vals = _uniq[c][0].to_list()
        # 0-based so the label matches the feature-matrix code exactly:
        # Xmatch/Xsub code k IS <prefix>k, no offset.
        _maps[c] = {h: f"{ANON_LABEL[c]}{i}" for i, h in enumerate(_vals)}
    _lf_anon = _lf_anon.with_columns(
        [pl.col(c).replace_strict(_maps[c], default=None).alias(c) for c in _label_cols]
    )

# High-cardinality columns: dense rank only, no label.
for c in _dense_cols:
    _lf_anon = _lf_anon.with_columns(
        pl.col(c).rank("dense").cast(pl.Int32).alias(c)
    )

_scrubbed = [c for c in ANON_SCRUB if c in _cols]
for c in _scrubbed:
    _lf_anon = _lf_anon.with_columns(
        pl.col(c).cast(pl.Utf8)
          .str.replace_all(r"(?i)(by user )\S+", r"${1}<redacted>")
          .str.replace_all(r"(?i)(user )[A-Za-z0-9._-]+( has removed)", r"${1}<redacted>${2}")
          .alias(c)
    )

# Restrict to the requested submission window.
_m0 = _dt.datetime.fromisoformat(ANON_START).replace(tzinfo=_dt.timezone.utc)
_m1 = _dt.datetime.fromisoformat(ANON_END).replace(tzinfo=_dt.timezone.utc)
_lf_anon = _lf_anon.filter(
    (to_sec(qcol) >= _m0.timestamp()) & (to_sec(qcol) < _m1.timestamp())
)

os.makedirs(os.path.dirname(ANON_OUT), exist_ok=True)
_lf_anon.sink_parquet(ANON_OUT, compression="zstd", compression_level=10)

print(f"Wrote {ANON_OUT}")
print(f"  window  : [{_m0:%Y-%m-%d} .. {_m1:%Y-%m-%d})")
print(f"  labelled: " + ", ".join(f"{c}->{ANON_LABEL[c]}N" for c in _label_cols))
print(f"  dense   : {', '.join(_dense_cols)}  (too many distinct values to label)")
print(f"  NOTE    : Xmatch/Xsub code k == <prefix>k here (0-based, exact match)")
print(f"  scrubbed: {', '.join(_scrubbed)}")
print(f"  readable: {', '.join(c for c in ANON_KEEP if c in _cols)}")

_chk = pl.scan_parquet(ANON_OUT)
print(f"  rows    : {_chk.select(pl.len()).collect().item():,}")
print(f"  size    : {os.path.getsize(ANON_OUT) / 1e6:,.1f} MB")
# Fail loudly if anything identifying survived.
_leaks = []
_csch = _chk.collect_schema()
for c in _label_cols:
    _sample = _chk.select(pl.col(c)).drop_nulls().head(1).collect()[c].to_list()
    if _sample and not str(_sample[0]).startswith(ANON_LABEL[c]):
        _leaks.append(f"{c} holds {_sample[0]!r}, not a {ANON_LABEL[c]}N pseudonym")
for c in _dense_cols:
    if _csch.get(c) != pl.Int32:
        _leaks.append(f"{c} is {_csch.get(c)}, expected dense Int32")
for c in _scrubbed:
    # Polars' regex engine has no look-around, so count-and-subtract instead.
    _tot = _chk.filter(pl.col(c).str.contains(r"(?i)by user ")).select(pl.len()).collect().item()
    _red = _chk.filter(pl.col(c).str.contains(r"(?i)by user <redacted>")).select(pl.len()).collect().item()
    if _tot - _red:
        _leaks.append(f"{c} still has {_tot - _red:,} un-redacted 'by user' values")
print("  CHECK   : " + ("OK, no identifying values found" if not _leaks else "LEAKS -> " + "; ".join(_leaks)))

### Saving training data for later runs (skip)

In [8]:
import os 
import json
import numpy as np

# Same root the harness reads; override with FIFE_DATA_ROOT.
SAVE_DIR = os.environ.get("FIFE_DATA_ROOT",
                          "/mnt/scratch/fast0/amaustin/datasets/fife")
os.makedirs(SAVE_DIR, exist_ok=True)

# 1. Save target arrays individually for fast loading
np.save(os.path.join(SAVE_DIR, "failed.npy"), failed)
np.save(os.path.join(SAVE_DIR, "fault_type.npy"), ftype)
np.save(os.path.join(SAVE_DIR, "hw.npy"), hw)
# E1/E3 target mask: jobs that actually started. Never-ran rows remain in the
# feature matrices as queue context but are not valid prediction targets.
np.save(os.path.join(SAVE_DIR, "ran.npy"), ran)

# 2. Update the targets in the .npz file (without saving Xmatch/Xsub)
np.savez_compressed(
    os.path.join(SAVE_DIR, "targets_and_masks.npz"),
    failed=failed,
    hw=hw,
    wait_sv=wait_sv,
    tr_mask=tr_mask,
    te_mask=te_mask,
    qs=qs,
    jst=jst,
    comp=comp,
    wall=wall,
    ran=ran,
    qexit=qexit,
    fault_type=ftype
)

# 3. Sanity check directly from disk
saved_failed = np.load(os.path.join(SAVE_DIR, "failed.npy"))
print(f"Saved failed.npy successfully.")
print(f"Total rows       : {len(saved_failed):,}")
print(f"Rows (all)       : {len(ran):,}   target population (Ran): {int(ran.sum()):,}")
print(f"Genuine Failures : {int((saved_failed[ran]==1).sum()):,} (within the target population)")
print(f"Hardware Faults  : {hw.sum():,} ({hw.sum()/max(saved_failed.sum(),1)*100:.2f}% of failures)")
_c, _j, _w = np.isfinite(comp), np.isfinite(jst), np.isfinite(wall)
print(f"Terminal time    : CompletionDate {_c.sum():,} | start+wall "
      f"{int((~_c & _j & _w).sum()):,} | neither {int((~_c & ~(_j & _w)).sum()):,}")

Saved failed.npy successfully.
Total rows       : 64,088,358
Rows (all)       : 64,088,358   target population (Ran): 56,918,813
Genuine Failures : 8,603,725 (within the target population)
Hardware Faults  : 752,379 (8.33% of failures)
Terminal time    : CompletionDate 55,264,860 | start+wall 1,652,885 | neither 7,170,613


Saving metadata

In [9]:
metadata = {
    "XMATCH_COLS": XMATCH_COLS,
    "XSUB_COLS": XSUB_COLS,
    "cards": [int(c) for c in cards],
    "NCAT_MATCH": len(CAT_ALL),
    "NCAT_SUB": len(SUB_CAT),
    "n_base": n_base,
    # Per-key trailing windows, so a saved matrix records which
    # windows its trailing columns were built with.
    "TRAIL_WINDOWS_BY_KEY": TRAIL_WINDOWS_BY_KEY,
}

# Code -> label for CampaignType (JSON object keys must be strings). The
# codes come from a hash + dense-rank, so they carry no inherent meaning --
# without this map the column is unreadable downstream.
if "CAMPAIGN_TYPE_CODES" in dir():
    metadata["CAMPAIGN_TYPE_CODES"] = {str(k): v for k, v in CAMPAIGN_TYPE_CODES.items()}
else:
    print("[!] CAMPAIGN_TYPE_CODES not defined -- run the CampaignType derivation cell.")

with open(os.path.join(SAVE_DIR, "schema_meta.json"), "w") as f:
    json.dump(metadata, f, indent=2)